In [ ]:
# loading in libraries
import json
from pathlib import Path
from pprint import pprint
from langchain_community.document_loaders import JSONLoader
import os
import getpass
import ApiKeyData
from langchain_openai import ChatOpenAI
#from langchain_openai import OpenAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_community.vectorstores import Chroma
import chromadb
import runhouse as rh
#from langchain_community.embeddings import SelfHostedHuggingFaceEmbeddings
from sentence_transformers import SentenceTransformer
from langchain.schema import Document

In [ ]:
#setting up model
llm = ChatOpenAI(openai_api_key=ApiKeyData.LANGCHAIN_API_KEY, temperature=0.7)
#temperature helps with creativeness of the responses the smaller the temperature the more direct the response will be

# add extra_body if I want to add to the current json data

In [ ]:
#embeddings = OpenAIEmbeddings(model="text-embedding-3-large") # <- better data but costly to implement
#embeddings = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2") # <- cost effective option

# gpu = rh.cluster(name="rh-a10x", instance_type="A100:1") #creates a cluster that helps with high-performance computing and deep learning tasks - helps accelerate embedding task
# embeddings = SelfHostedHuggingFaceEmbeddings(model_id=model_id, hardware=gpu)

In [ ]:
# loading in data
file_path = './winemag-data-130k-v2.json'
data = json.loads(Path(file_path).read_text())
# will print data

In [ ]:
print(data[0])

In [ ]:
# CREATING DOCUMENTS SO DATA CAN BE EMBEDDED
documents = []
metadatas = []
i = 0
for entry in data:
    # Construct each document
    doc = Document(
        page_content=entry.get('description', 'variety'),  # or another field like 'title'
        # metadata=entry 
    )
    documents.append(doc)
    metadatas.append(entry) # You can store all other fields in metadata for later use

#making it usable for Chromadb
documentString = [item.page_content for item in documents]

# Verify the number of documents created
print(f"Number of documents created: {len(documents)}")

In [ ]:
# LOADING DATA
#NOT WORKING
# loader = JSONLoader(file_path="./winemag-data-130k-v2.json", jq_schema=".", text_content=False)
# documents = loader.load() #loads data into document objects

# jq_schema="." extracts data from the whole json file not just a single array
# text_content=False tells model that this is structured data not raw text

In [ ]:
# EMBEDDING
texts = [doc.page_content for doc in documents] #extracts text for embedding

model = SentenceTransformer('all-MiniLM-L6-v2') #same model that Chromadb uses - intializing embedding model

document_embeddings = model.encode(texts) #converts text into numerical embeddings - converts each document into a vector representation (embedding)

In [ ]:
print(len(ids))
print(len(document_embeddings))
print(metadatas[0])
# metadataString = [item.metadata for item in metadatas] <- ignore this for now

In [ ]:
# VECTORIZING DATA
# chroma_client.delete_collection(name="wine_data_collection") #will clear df as needed

chroma_client = chromadb.Client() # inializes chromadb so we can connect our collection to the vector storage

collection = chroma_client.create_collection(name="Wine_Data_Collection") #creating collection - this is essentially a vector db that can store embeddings, queries, and documents
# NOTE - The collection name must start and end with a lowercase letter
ids = [f"doc_{i}" for i in range(len(document_embeddings[0]))]

collection.add(
    ids=ids,  #list of unqiue ids connected to documents
    embeddings=document_embeddings,  # List of embeddings
    # metadatas=None, #optional
    documents=documentString #optional
)

# retriever = db.as_retriever()

# If Chroma is passed a list of documents, it will automatically tokenize and embed them with the
#  collection's embedding function (the default will be used if none was supplied at collection creation)
# Chroma will also store the documents themselves.

In [ ]:
# Lets me peek at collection to ensure that everything is loading in properly
print(collection.peek())

In [ ]:
# Querying model to search for this type of wine
results = collection.query(
    query_texts=["Can you show me a smoky wine?"],
    n_results=3,
    #include=["documents"]
)
results